# B2.6 · Sandbox replication

**Function B — Application Security with an AI SDLC → The AI SDLC: an Agentic AppSec Pipeline**  ·  *Security of AI*

Builds on **[B2.5 · Feasibility filtering and reachability](https://spbreed.github.io/cyber-commons/lessons/B2.5.html)**.

| | |
|---|---|
| Tools used | Docker, gVisor, Cilium, GLM-4.6, Claude Haiku 4.5 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

You cannot exploit a finding to confirm it without somewhere safe to do it. The replica is that place, and the fidelity you give it decides which findings you are able to confirm at all.

> **At CyberTravels.** You cannot confirm the IDOR by exploiting it in production. The replica is where the booking API can be attacked safely, and its fidelity decides which findings are confirmable at all.

## 2 · The framework

```
   production            replica
   +-----------+         +------------------+
   | real data |   -->   | stubbed data     |
   | real deps |         | recorded deps    |
   | real users|         | nobody           |
   +-----------+         +------------------+
                                 |
                        exploit here, safely, on purpose

   fidelity decides which findings you can confirm at all
```

Phase 4 turns hypotheses into facts by running the application. That is only
safe if the thing you run it against cannot hurt anyone.

**Stage 11 — Sandbox replication.** Deploy the application in an isolated,
disposable runtime: its own container, its own synthetic data, no route to
production, no real credentials.

The reason this is a *stage* rather than a footnote is that the obvious shortcut
— point the dynamic tests at staging — converts every destructive probe into an
incident. Staging usually shares an identity provider, a message bus, sometimes
a database replica, and always someone's on-call rota.

Four isolation properties, and you need all four:

- **network** — no egress except to the replica itself,
- **credentials** — synthetic secrets, so a leak is worthless,
- **data** — synthetic records, so an exfiltration test exfiltrates nothing,
- **lifetime** — destroyed after the run, so state cannot leak between tests.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · Stage 11 — model the replica and its isolation

In [ ]:
import re
from dataclasses import dataclass, field
from urllib.parse import urlparse

PRIVATE = [re.compile(p) for p in (r"^127\.", r"^10\.", r"^169\.254\.",
                                   r"^192\.168\.", r"^localhost$")]

@dataclass
class Sandbox:
    name: str
    allow_hosts: set = field(default_factory=set)
    credentials: dict = field(default_factory=dict)
    data: dict = field(default_factory=dict)
    ephemeral: bool = True
    destroyed: bool = False
    log: list = field(default_factory=list)

    def egress(self, url):
        host = (urlparse(url).hostname or "").lower()
        if host in self.allow_hosts:
            d = (True, "replica-internal")
        elif any(p.match(host) for p in PRIVATE):
            d = (False, "private address outside the replica — blocked")
        else:
            d = (False, "not on the replica allowlist")
        self.log.append((url, d[0], d[1])); return d

    def destroy(self):
        self.credentials.clear(); self.data.clear(); self.destroyed = True
        return "replica destroyed; state cannot leak into the next run"

REPLICA = Sandbox("appsec-replica-8812",
                  allow_hosts={"replica.local", "db.replica.local"},
                  credentials={"DB_PASSWORD": "synthetic-not-real-0000",
                               "API_TOKEN": "synthetic-not-real-1111"},
                  data={"users": [{"id": 1, "name": "test-user-a",
                                   "card": "4000000000000000"}]})

for url in ["http://replica.local/reports",
            "http://db.replica.local:5432/",
            "https://api.github.com/",
            "http://169.254.169.254/latest/meta-data/",
            "http://10.0.3.14:9200/_search"]:
    ok, why = REPLICA.egress(url)
    print(f"{'ALLOW' if ok else 'DENY ':5s} {url[:44]:46s} {why}")

## 4 · Where it breaks — the same probes against staging

In [ ]:
STAGING = Sandbox("staging",
                  allow_hosts={"staging.internal", "db.staging.internal",
                               "sso.corp", "bus.corp", "api.github.com"},
                  credentials={"DB_PASSWORD": "real-staging-password",
                               "API_TOKEN": "ghp_real_staging_token"},
                  data={"users": [{"id": 4471, "name": "dana@corp",
                                   "card": "4111111111111111"}]},
                  ephemeral=False)

DESTRUCTIVE_PROBES = [
 ("drop a table",        "db.staging.internal", "db.replica.local"),
 ("exfiltrate user rows","api.github.com",      "replica.local"),
 ("brute-force login",   "sso.corp",            "replica.local"),
]
print(f"{'probe':24s}{'on staging':>14}{'on replica':>14}")
print("-" * 56)
for probe, staging_target, replica_target in DESTRUCTIVE_PROBES:
    s_ok, _ = STAGING.egress(f"http://{staging_target}/")
    r_ok, _ = REPLICA.egress(f"http://{replica_target}/")
    print(f"{probe:24s}{'REACHES':>14}{'contained':>14}" if s_ok
          else f"{probe:24s}{'blocked':>14}{'contained':>14}")

print("\nwhat a leaked credential is worth:")
for name, box in (("staging", STAGING), ("replica", REPLICA)):
    creds = list(box.credentials.values())
    synthetic = all("synthetic" in c for c in creds)
    print(f"   {name:10s}{'worthless — synthetic' if synthetic else 'REAL — usable against real systems'}")

print("\nwhat an exfiltrated record is worth:")
for name, box in (("staging", STAGING), ("replica", REPLICA)):
    rec = box.data["users"][0]
    real = not rec["name"].startswith("test-")
    print(f"   {name:10s}{'REAL customer data' if real else 'synthetic'}  {rec['name']}")

## 5 · The control — the four isolation properties, checked

In [ ]:
def isolation_report(box):
    checks = {
      "network": all(not ok for url, ok, _ in box.log
                     if urlparse(url).hostname not in box.allow_hosts),
      "credentials": all("synthetic" in v for v in box.credentials.values()) if box.credentials else True,
      "data": all(u["name"].startswith("test-") for u in box.data.get("users", [])),
      "lifetime": box.ephemeral,
    }
    return checks, all(checks.values())

for box in (REPLICA, STAGING):
    checks, ok = isolation_report(box)
    print(f"{box.name}")
    for k, v in checks.items():
        print(f"   {'PASS' if v else 'FAIL':5s} {k}")
    print(f"   → suitable for dynamic testing: {ok}\n")

_, replica_ok = isolation_report(REPLICA)
_, staging_ok = isolation_report(STAGING)
assert replica_ok and not staging_ok

print(REPLICA.destroy())
print(f"credentials after destroy: {REPLICA.credentials or 'cleared'}")
print(f"data after destroy:        {REPLICA.data or 'cleared'}")
assert REPLICA.destroyed and not REPLICA.credentials

## What you just proved

The replica permits only its own internal hosts and blocks GitHub, the metadata service and private addresses. Staging holds real credentials and a real-shaped customer record while the replica holds synthetic ones. The four isolation checks pass for the replica and fail for staging on credentials, data and lifetime, and destroying the replica clears its state.

## Your turn

Check whether your dynamic testing currently runs against staging. If it does, list what staging shares with production — identity provider, message bus, data replica. Each shared component is a path from a test probe to a real incident.

---

**Next → [B2.7 · Dynamic exploitation (DAST)](https://spbreed.github.io/cyber-commons/lessons/B2.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*